In [15]:
import numpy as np
import pandas as pd
from scipy import stats
import sys
sys.path.insert(0, '../')
from ultility.metrics import rolling_std_vol
from ultility.models_garch import garch_forecast_fixed_params
from ultility.models_lstm_baseline import train_lstm_baseline, predict_lstm_baseline_rolling
from ultility.models_transformer import train_transformer
from ultility.lstmgarch import LSTMGARCH
from ultility.transformer_garch import TransformerGARCH
from ultility.eval_vol import save_and_plot
import torch
import torch.nn as nn
import time
import warnings
warnings.filterwarnings('ignore')
import ultility.var_calculator as var_calculator

In [24]:
from pathlib import Path
import copy

import numpy as np
import pandas as pd
from scipy import stats
import torch
import time

from ultility.metrics import compute_mse_qlike
from ultility.models_garch import garch_forecast_fixed_params
from ultility.models_lstm_baseline import train_lstm_baseline, predict_lstm_baseline_rolling
from ultility.models_transformer import train_transformer
from ultility.lstmgarch import LSTMGARCH
from ultility.transformer_garch import TransformerGARCH
import ultility.var_calculator as var_calculator


In [25]:
datasets = {}
for csv_name, key in [('VN30_INDEX.csv', 'VN30 Index'), ('VN_INDEX.csv', 'VN Index')]:
    df_temp = pd.read_csv(f'../dataset/{csv_name}')
    df_temp['time'] = pd.to_datetime(df_temp['time'], format='mixed', dayfirst=True, errors='coerce')
    df_temp = df_temp.sort_values('time')
    ret_col = 'return_1_day'
    df_temp_filtered = df_temp[df_temp['time'].dt.year >= 2010]
    null_count = df_temp_filtered[ret_col].isna().sum()
    if null_count > 0:
        for idx, val in df_temp_filtered[df_temp_filtered[ret_col].isna()][ret_col].items():
            print(f"{key}: null at {df_temp.loc[idx, 'time'].strftime('%Y-%m-%d')}")
    datasets[key] = df_temp_filtered.set_index('time')[ret_col] * 100

for file in ['DAX_40.csv', 'EuroNext_100.csv', 'IBEX_35.csv', 'KOSPI_index.csv', 'SMI.csv', 'snp500.csv', 'Nikkei_225.csv']:
    df_temp = pd.read_csv(f'../dataset/{file}')
    if 'Date' in df_temp.columns:
        df_temp.rename(columns={'Date': 'time'}, inplace=True)
    df_temp['time'] = pd.to_datetime(df_temp['time'], format='mixed', dayfirst=False, errors='coerce')
    df_temp = df_temp.dropna(subset=['time']).sort_values('time')
    ret_col = 'return_1_day'
    df_temp_filtered = df_temp[df_temp['time'].dt.year >= 2010]
    null_count = df_temp_filtered[ret_col].isna().sum()
    if null_count > 0:
        for idx, val in df_temp_filtered[df_temp_filtered[ret_col].isna()][ret_col].items():
            print(f"{file}: null at {df_temp.loc[idx, 'time'].strftime('%Y-%m-%d')}")
    datasets[file.replace('.csv', '')] = df_temp_filtered.set_index('time')[ret_col] * 100


In [26]:
#Kết quả từ KS split
summary_split = pd.DataFrame({
    "Dataset": [
        "VN30 Index", "VN Index", "DAX_40", "EuroNext_100", "IBEX_35", "KOSPI_index", "SMI", "snp500", "Nikkei_225"
    ],
    "train_size": [1971, 1964, 1983, 2005, 1815, 1944, 1987, 1988, 1677],
    "val_size":   [1096, 1103, 1104, 1115, 1362, 1109, 1135, 1109, 1174],
    "test_size":  [925, 925, 972, 979, 923, 881, 901, 927, 1062],
    "split_i":    [1971, 1964, 1983, 2005, 1815, 1944, 1987, 1988, 1677],
    "split_j":    [3067, 3067, 3087, 3120, 3177, 3053, 3122, 3097, 2851]
})
split_df = summary_split.set_index("Dataset")[["train_size", "val_size", "test_size"]]
split_df.columns = ["Train", "Val", "Test"]
split_df

,Train,Val,Test
Dataset,,,
VN30 Index,1971,1096,925
VN Index,1964,1103,925
DAX_40,1983,1104,972
EuroNext_100,2005,1115,979
IBEX_35,1815,1362,923
KOSPI_index,1944,1109,881
SMI,1987,1135,901
snp500,1988,1109,927
Nikkei_225,1677,1174,1062


In [27]:
def distribution_analysis(data, name):
    returns = pd.Series(data).dropna().values
    nu, loc, scale = stats.t.fit(returns)
    ks_stat, ks_p = stats.kstest(returns, "t", args=(nu, loc, scale))
    norm_loc, norm_scale = stats.norm.fit(returns)
    ks_stat_norm, ks_p_norm = stats.kstest(returns, "norm", args=(norm_loc, norm_scale))
    return nu, loc, scale, ks_stat, ks_p, ks_stat_norm, ks_p_norm



In [28]:
res = []; 
for ds in split_df.index:
    tr, va, te = int(split_df.loc[ds, 'Train']), int(split_df.loc[ds, 'Val']), int(split_df.loc[ds, 'Test'])
    s = datasets[ds].values
    res.append({'Dataset': ds, 'nu_train': distribution_analysis(s[:tr], '')[0], 'nu_val': distribution_analysis(s[tr:tr+va], '')[0], 'nu_test': distribution_analysis(s[tr+va:tr+va+te], '')[0], 'nu_all': distribution_analysis(s[:tr+va+te], '')[0]})
pd.DataFrame(res)

,Dataset,nu_train,nu_val,nu_test,nu_all
0,VN30 Index,4.696197,2.696639,2.385055,3.251474
1,VN Index,4.573257,2.497934,2.509615,3.181560
2,DAX_40,3.424473,2.728275,4.298158,3.232514
3,EuroNext_100,3.502448,2.609397,4.072105,3.203942
4,IBEX_35,4.400113,3.303941,5.962785,3.587792
5,KOSPI_index,3.426984,3.772418,5.169135,3.754324
6,SMI,3.589687,3.226405,4.724666,3.603772
7,snp500,2.705923,2.434559,3.422548,2.715517
8,Nikkei_225,4.926478,3.030612,4.596454,4.053247


In [ ]:

output_dir = Path('../output')
output_dir.mkdir(parents=True, exist_ok=True)

seq_len = 60
horizons = [1, 2, 5, 10, 20]
confidence_level = 0.95


def create_sequences(data, seq_len):
    sequences = []
    for index in range(len(data) - seq_len):
        sequences.append(data[index:index + seq_len])
    return np.asarray(sequences, dtype=np.float32)


def get_device():
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def train_lstm_garch_with_validation(train_data, val_data, test_data, hidden_dim=16, epochs=120, batch_size=64, learning_rate=1e-3, lr_factor=0.5, lr_patience=5, early_stopping_patience=20, min_lr=1e-6):
    device = get_device()
    train_seq = create_sequences(train_data, seq_len)
    if len(train_seq) == 0:
        raise ValueError('Not enough data for LSTM-GARCH sequences')

    train_loader = torch.utils.data.DataLoader(
        torch.tensor(train_seq, dtype=torch.float32),
        batch_size=batch_size,
        shuffle=True,
        pin_memory=(device.type == 'cuda'),
    )
    model = LSTMGARCH(hidden_dim=hidden_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=lr_factor,
        patience=lr_patience,
        min_lr=min_lr,
    )

    if val_data is not None and len(val_data) > 0:
        context = np.concatenate([train_data[-seq_len:], val_data]) if len(train_data) >= seq_len else np.concatenate([train_data, val_data])
        val_seq = create_sequences(context, seq_len)
    else:
        val_seq = None

    best_state = None
    best_val_loss = float('inf')
    patience = 0

    for _ in range(epochs):
        model.train()
        train_losses = []
        for batch in train_loader:
            batch = batch.to(device, non_blocking=True)
            optimizer.zero_grad()
            loss, _ = model(batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())

        train_loss = float(np.mean(train_losses)) if train_losses else float('inf')
        if val_seq is not None and len(val_seq) > 0:
            val_loader = torch.utils.data.DataLoader(
                torch.tensor(val_seq, dtype=torch.float32),
                batch_size=batch_size,
                shuffle=False,
                pin_memory=(device.type == 'cuda'),
            )
            val_losses = []
            model.eval()
            with torch.no_grad():
                for batch in val_loader:
                    batch = batch.to(device, non_blocking=True)
                    val_loss, _ = model(batch)
                    val_losses.append(val_loss.item())
            val_loss = float(np.mean(val_losses)) if val_losses else train_loss
        else:
            val_loss = train_loss

        scheduler.step(val_loss)
        if val_loss < best_val_loss - 1e-6:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience = 0
        else:
            patience += 1

        if patience >= early_stopping_patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    history = list(train_data)
    preds = []
    with torch.no_grad():
        for value in test_data:
            if len(history) < seq_len:
                history.append(value)
                continue
            seq = torch.tensor(history[-seq_len:], dtype=torch.float32, device=device).unsqueeze(0)
            _, sigma2 = model(seq)
            preds.append(torch.sqrt(sigma2[:, -1]).cpu().numpy()[0])
            history.append(value)
    return np.asarray(preds, dtype=float)


def train_transformer_garch(train_data, test_data, epochs=80, batch_size=64, learning_rate=1e-3):
    device = get_device()
    train_seq = create_sequences(train_data, seq_len)
    if len(train_seq) == 0:
        raise ValueError('Not enough data for Transformer-GARCH sequences')

    train_loader = torch.utils.data.DataLoader(
        torch.tensor(train_seq, dtype=torch.float32),
        batch_size=batch_size,
        shuffle=True,
        pin_memory=(device.type == 'cuda'),
    )
    model = TransformerGARCH().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    for _ in range(epochs):
        model.train()
        for batch in train_loader:
            batch = batch.to(device, non_blocking=True)
            optimizer.zero_grad()
            loss, _ = model(batch)
            loss.backward()
            optimizer.step()

    model.eval()
    history = list(train_data)
    preds = []
    with torch.no_grad():
        for value in test_data:
            if len(history) < seq_len:
                history.append(value)
                continue
            seq = torch.tensor(history[-seq_len:], dtype=torch.float32, device=device).unsqueeze(0)
            _, sigma2 = model(seq)
            preds.append(torch.sqrt(sigma2[:, -1]).cpu().numpy()[0])
            history.append(value)
    return np.asarray(preds, dtype=float)


model_names = ['GARCH', 'GJR-GARCH', 'Transformer', 'LSTM-Baseline', 'LSTM-GARCH', 'Transformer-GARCH']
predictions = {model_name: {} for model_name in model_names}
training_times = {model_name: {} for model_name in model_names}

for ds in split_df.index:
    series = datasets[ds]
    if not isinstance(series, pd.Series):
        series = pd.Series(np.asarray(series))

    tr, va, te = map(int, split_df.loc[ds, ['Train', 'Val', 'Test']])
    train_data = pd.to_numeric(series.iloc[:tr].values, errors='coerce')
    val_data = pd.to_numeric(series.iloc[tr:tr + va].values, errors='coerce')
    test_data = pd.to_numeric(series.iloc[tr + va:tr + va + te].values, errors='coerce')

    start = time.perf_counter()
    predictions['GARCH'][ds] = garch_forecast_fixed_params(train_data, test_data, model_name='GARCH')
    training_times['GARCH'][ds] = (time.perf_counter() - start) / 60

    start = time.perf_counter()
    predictions['GJR-GARCH'][ds] = garch_forecast_fixed_params(train_data, test_data, model_name='GJR-GARCH')
    training_times['GJR-GARCH'][ds] = (time.perf_counter() - start) / 60

    start = time.perf_counter()
    predictions['Transformer'][ds] = np.abs(train_transformer(np.abs(train_data), np.abs(test_data), seq_len=seq_len, epochs=100))
    training_times['Transformer'][ds] = (time.perf_counter() - start) / 60

    start = time.perf_counter()
    lstm_model, lstm_scaler, _, _ = train_lstm_baseline(train_data, val_data, seq_len=seq_len, epochs=80, batch_size=32, learning_rate=1e-3, device=str(get_device()))
    predictions['LSTM-Baseline'][ds] = predict_lstm_baseline_rolling(lstm_model, train_data, test_data, seq_len=seq_len, scaler=lstm_scaler, device=str(get_device()))
    training_times['LSTM-Baseline'][ds] = (time.perf_counter() - start) / 60

    start = time.perf_counter()
    predictions['LSTM-GARCH'][ds] = train_lstm_garch_with_validation(train_data, val_data, test_data, hidden_dim=16, epochs=120)
    training_times['LSTM-GARCH'][ds] = (time.perf_counter() - start) / 60

    start = time.perf_counter()
    predictions['Transformer-GARCH'][ds] = train_transformer_garch(train_data, test_data, epochs=80)
    training_times['Transformer-GARCH'][ds] = (time.perf_counter() - start) / 60


horizon_rows = []
horizon_result_rows = []
for ds in split_df.index:
    series = datasets[ds]
    if not isinstance(series, pd.Series):
        series = pd.Series(np.asarray(series))

    tr, va, te = map(int, split_df.loc[ds, ['Train', 'Val', 'Test']])
    test_series = series.iloc[tr + va:tr + va + te]
    test_returns = pd.to_numeric(test_series.values, errors='coerce')
    test_dates = pd.to_datetime(test_series.index, errors='coerce')

    nu = distribution_analysis(series.iloc[:tr], '')[0]
    if pd.isna(nu) or nu <= 2:
        nu = 8.0
    scale = np.sqrt((nu - 2) / nu)
    t_alpha = -stats.t.ppf(1 - confidence_level, df=nu)

    for model_name, model_preds in predictions.items():
        base_pred = np.asarray(model_preds.get(ds, []), dtype=float).reshape(-1)
        if base_pred.size == 0:
            continue

        train_time_min = training_times[model_name].get(ds, np.nan)

        for horizon in horizons:
            n_eval = min(base_pred.size, len(test_returns) - horizon)
            if n_eval <= 0:
                continue

            origin_dates = test_dates[:n_eval]
            target_dates = test_dates[horizon:horizon + n_eval]
            future_returns = np.array([np.nansum(test_returns[i + 1:i + 1 + horizon]) for i in range(n_eval)], dtype=float)
            realized_vol_h = np.array([
                np.sqrt(np.nansum(np.square(test_returns[i + 1:i + 1 + horizon])))
                for i in range(n_eval)
            ], dtype=float)
            pred_vol_h = base_pred[:n_eval] * np.sqrt(horizon)
            var_h = scale * t_alpha * pred_vol_h

            mse_h, qlike_h = compute_mse_qlike(realized_vol_h, pred_vol_h)
            violation_rate, kupiec_lr, kupiec_p = var_calculator.compute_kupiec_test(
                future_returns,
                var_h,
                confidence_level=confidence_level,
            )
            traffic_light, cum_violations = var_calculator.compute_traffic_light_test(
                future_returns,
                var_h,
                confidence_level=confidence_level,
            )
            lr_ind, p_ind = var_calculator.compute_christoffersen_independence(
                future_returns,
                var_h,
                confidence_level=confidence_level,
            )

            for origin_date, target_date, future_ret, realized_vol, pred_vol, var_value in zip(
                origin_dates,
                target_dates,
                future_returns,
                realized_vol_h,
                pred_vol_h,
                var_h,
            ):
                horizon_rows.append({
                    'Dataset': ds,
                    'Model': model_name,
                    'Horizon': horizon,
                    'Origin_Date': origin_date,
                    'Target_Date': target_date,
                    'return_h': float(future_ret) if pd.notna(future_ret) else np.nan,
                    'realized_vol_h': float(realized_vol) if pd.notna(realized_vol) else np.nan,
                    'predicted_vol_h': float(pred_vol) if pd.notna(pred_vol) else np.nan,
                    'var_h': float(var_value) if pd.notna(var_value) else np.nan,
                    'Train_Time_min': train_time_min,
                })

            horizon_result_rows.append({
                'Dataset': ds,
                'Model': model_name,
                'Horizon': horizon,
                'MSE': mse_h,
                'QLIKE': qlike_h,
                'Violation_Rate': violation_rate,
                'Kupiec_LR': kupiec_lr,
                'Kupiec_p': kupiec_p,
                'LR_Ind': lr_ind,
                'p_Ind': p_ind,
                'Traffic_Light': traffic_light,
                'Cum_Violations': cum_violations,
                'Train_Time_min': train_time_min,
                'nu_train': nu,
            })

horizon_preds_df = pd.DataFrame(horizon_rows)
horizon_results_df = pd.DataFrame(horizon_result_rows)

if not horizon_preds_df.empty:
    horizon_preds_df['Origin_Date'] = pd.to_datetime(horizon_preds_df['Origin_Date'], errors='coerce')
    horizon_preds_df['Target_Date'] = pd.to_datetime(horizon_preds_df['Target_Date'], errors='coerce')
    horizon_preds_df = horizon_preds_df.sort_values(['Dataset', 'Model', 'Horizon', 'Origin_Date']).reset_index(drop=True)
    horizon_preds_df.to_csv(output_dir / 'predicts_horizon.csv', index=False)

if not horizon_results_df.empty:
    horizon_results_df = horizon_results_df.sort_values(['Dataset', 'Model', 'Horizon']).reset_index(drop=True)
    horizon_results_df.to_csv(output_dir / 'models_results_horizon.csv', index=False)

print(horizon_results_df.head(20).to_string(index=False))